In [17]:
import pandas as pd
import os
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv, find_dotenv
from sentence_transformers import SentenceTransformer

In [3]:
files = pd.read_csv("course_descriptions.csv", encoding="ANSI")

In [4]:
def create_course_description(row):
    return f'''The course name is {row["course_name"]}, the slug is {row["course_slug"]},
    the tecchnology is {row["course_technology"]} and the course topic is {row["course_topic"]}'''

In [5]:
pd.set_option('display.max_rows', 106)
files['course_description_new'] = files.apply(create_course_description,axis=1)
print(files["course_description_new"])

0      The course name is Introduction to Tableau, th...
1      The course name is The Complete Data Visualiza...
2      The course name is Introduction to R Programmi...
3      The course name is Data Preprocessing with Num...
4      The course name is Introduction to Data and Da...
5      The course name is Data Cleaning and Preproces...
6      The course name is Introduction to Business An...
7      The course name is Data Analysis with Excel Pi...
8      The course name is SQL, the slug is sql,\n    ...
9      The course name is Credit Risk Modeling in Pyt...
10     The course name is Python Programmer Bootcamp,...
11     The course name is SQL + Tableau + Python, the...
12     The course name is Introduction to Jupyter, th...
13     The course name is Statistics, the slug is sta...
14     The course name is Mathematics, the slug is ma...
15     The course name is Introduction to Excel, the ...
16     The course name is Probability, the slug is pr...
17     The course name is Start

In [2]:
%load_ext dotenv
%dotenv

In [6]:
load_dotenv(find_dotenv(), override = True)

True

In [10]:
pc = Pinecone(api_key= os.environ.get('PINECONE_API_KEY'), environment = os.environ.get("PINECONE_ENV"))

In [11]:
index_name = "my-index"
dimension = 384
metric = "cosine"

In [13]:
if index_name in [index.name for index in pc.list_indexes()]:
    pc.delete_index(index_name)
    print(f"{index_name} succesfully deleted.")
else:
    print(f"{index_name} not in index list.")

my-index succesfully deleted.


In [15]:
pc.create_index(
    name = index_name,
    dimension = dimension,
    metric = metric,
    spec = ServerlessSpec(
        cloud = "aws",
        region = "us-east-1"
    )
)

{
    "name": "my-index",
    "metric": "cosine",
    "host": "my-index-v7kbncc.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "vary": "origin, access-control-request-method, access-control-request-headers",
            "access-control-allow-origin": "*",
            "access-control-expose-headers": "*",
            "x-pinecone-api-version": "2025-10",


In [16]:
index = pc.Index(index_name)

C:\Users\HarshMistry\anaconda3\envs\langchain_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Embedding the data

In [18]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|█| 103/103 [00:00<00:00, 207.79it/s, Materializing
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [21]:
def create_embeddings(row):
    combined_text = ''.join([str(row[field]) for field in ['course_description','course_description_new','course_description_short']])
    embedding = model.encode(combined_text, show_progress_bar = False)
    return embedding

In [22]:
files["embedding"] = files.apply(create_embeddings, axis =  1)

In [26]:
vector_to_upsert = [(str(row["course_name"]),row["embedding"].tolist()) for _, row in files.iterrows()]
index.upsert(vectors = vector_to_upsert)

print("Data upserted to Pinecone index")

Data upserted to Pinecone index


# Semantic search

In [27]:
query = "Clustering"
query_embedding = model.encode(query, show_progress_bar = False).tolist()

In [29]:
query_results = index.query(
    vector = [query_embedding],
    top_k = 12,
    include_values = True
)

In [30]:
query_results

QueryResponse(matches=[{'id': 'Machine Learning in Excel',
 'score': 0.341700584,
 'values': [-0.00681828428,
            -0.0299020316,
            -0.0348869,
            -0.0120218173,
            -0.030609414,
            -0.0259555541,
            -0.0534992404,
            -0.0539726168,
            0.010541413,
            0.0346401408,
            -0.0349754319,
            -0.0548060946,
            0.066415824,
            -0.039986182,
            -0.00999202486,
            0.0382477343,
            0.00111681304,
            -0.00505274395,
            0.00319048925,
            -0.0694083795,
            0.0648970604,
            0.0410333946,
            -0.0509010777,
            -0.0251257028,
            0.0300838947,
            0.0179572161,
            0.0569334365,
            0.0082341861,
            -0.0396966413,
            -0.0267482754,
            -0.050261762,
            0.0162904784,
            0.00984217506,
            0.0505748838,
            -0.07

In [34]:
for match in query_results["matches"]:
    print(f"Matched item ID: {match['id']}, score:{match['score']}")

Matched item ID: Machine Learning in Excel, score:0.341700584
Matched item ID: Machine Learning with K-Nearest Neighbors, score:0.312284976
Matched item ID: Customer Churn Analysis with SQL and Tableau, score:0.283303291
Matched item ID: Machine Learning in Python, score:0.267083198
Matched item ID: Linear Algebra and Feature Selection, score:0.255191833
Matched item ID: Growth Analysis with SQL, Python, and Tableau  , score:0.254137069
Matched item ID: Fashion Analytics with Tableau, score:0.236253753
Matched item ID: Customer Engagement Analysis with SQL and Tableau, score:0.230439201
Matched item ID: Data Preprocessing with NumPy, score:0.215222374
Matched item ID: Data Analysis with Excel Pivot Tables, score:0.215018287
Matched item ID: Machine Learning with Naive Bayes, score:0.213718414
Matched item ID: Machine Learning with Support Vector Machines, score:0.207922
